# MCEQ · Classificação de notícias falsas no Fake.Br


Usando o mesmo TF-IDF e as mesmas notícias do Fake.Br, como o **F1 macro** do SVM linear se compara ao do Naive Bayes multinomial e ao da regressão logística?

A análise usa o corpus completo, incluindo todas as categorias temáticas. A hipótese principal é que o SVM linear obtém F1 macro maior que o Naive Bayes multinomial. A regressão logística entra como terceira referência. Todos os modelos usam o mesmo pré-processamento, representação e partições. Os parâmetros ficam fixos.

**Como reproduzir no Colab:** escolha **Ambiente de execução → Executar tudo**. A última etapa cria um ZIP com as tabelas, figuras e previsões. Resultados gravados abaixo correspondem à execução com a revisão oficial indicada no código.

## 1. Fonte e importação

O Fake.Br Corpus contém 7.200 notícias em português brasileiro, metade falsa e metade verdadeira. A versão de textos completos usa o mesmo nome de arquivo para as duas notícias de cada par. Guardamos o identificador do par, o texto e o rótulo. A entrada dos classificadores será formada só pelo texto.

[Repositório oficial do Fake.Br](https://github.com/roneysco/Fake.br-Corpus) · [Monteiro et al. (2018)](https://doi.org/10.1007/978-3-319-99722-3_33)

In [ ]:
import json
import platform
import re
import shutil
import subprocess
import unicodedata
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, average_precision_score,
    classification_report, ConfusionMatrixDisplay, confusion_matrix, f1_score,
    matthews_corrcoef, precision_score, recall_score, roc_auc_score,
    precision_recall_curve, roc_curve,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

semente = 42
revisao = "780f5516c4ae070761632d98ac3368f3ded09d35"
repo = Path("Fake.br-Corpus")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/roneysco/Fake.br-Corpus.git", str(repo)],
        check=True,
    )

git = ["git", "-C", str(repo)]
versao = subprocess.check_output([*git, "rev-parse", "HEAD"], text=True).strip()
if versao != revisao:
    subprocess.run([*git, "fetch", "--depth", "1", "origin", revisao], check=True)
    # Usa uma pasta própria para preservar um checkout preexistente.
    repo = Path(f"Fake.br-Corpus-{revisao[:8]}")
    if not repo.exists():
        subprocess.run([*git, "worktree", "add", "--detach", str(repo.resolve()), revisao], check=True)

noticias = pd.DataFrame([
    (arquivo.stem, rotulo, arquivo.read_text(encoding="utf-8-sig").strip())
    for rotulo in ("fake", "true")
    for arquivo in sorted((repo / "full_texts" / rotulo).glob("*.txt"))
], columns=["id_par", "rotulo", "texto"])
if noticias["rotulo"].value_counts().to_dict() != {"fake": 3600, "true": 3600}:
    raise ValueError("Esperavam-se 3.600 notícias de cada classe.")
contagem_pares = noticias.groupby(["id_par", "rotulo"]).size().unstack(fill_value=0)
if not contagem_pares.eq(1).all().all():
    raise ValueError("Cada par deve conter uma notícia de cada classe.")
print(f"Revisão do Fake.Br: {revisao}")
print(f"{len(noticias)} notícias e {len(contagem_pares)} pares completos.")

## 2. Limpeza e divisão dos dados

O texto vai para minúsculas, tem URLs explícitas e tags HTML removidas, perde os acentos e tem os símbolos trocados por espaços. Preservamos letras, números e palavras de negação. A limpeza é aplicada da mesma forma a todas as notícias.

Cada par permanece inteiro no treino ou no teste. Quando há textos idênticos após a limpeza, unimos seus pares no mesmo grupo de sorteio, mantendo todas as observações. Usamos a semente 42 e reservamos 720 pares (20%) para teste. Os outros 2.880 pares (80%) ficam no treino. A divisão mantém as classes equilibradas em ambas as partições.

In [ ]:
def limpar_texto(texto):
    texto = re.sub(r"https?://\S+|www\.\S+|<[^>]*>", " ", texto.lower())
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(letra for letra in texto if not unicodedata.combining(letra))
    return " ".join(re.sub(r"[^a-z0-9\s]", " ", texto).split())

noticias["texto_limpo"] = noticias["texto"].map(limpar_texto)
if noticias["texto_limpo"].eq("").any():
    raise ValueError("Há textos vazios após a limpeza.")

grupo_por_par = {par: par for par in contagem_pares.index}

def raiz(par):
    while grupo_por_par[par] != par:
        par = grupo_por_par[par]
    return par

duplicados = noticias.loc[noticias["texto_limpo"].duplicated(keep=False)]
for _, itens in duplicados.groupby("texto_limpo", sort=False):
    raizes = sorted({raiz(par) for par in itens["id_par"]})
    for outra_raiz in raizes[1:]:
        grupo_por_par[outra_raiz] = raizes[0]

noticias["grupo"] = noticias["id_par"].map(raiz)
noticias["classe"] = noticias["rotulo"].map({"true": 0, "fake": 1})

auditoria = pd.Series({
    "Notícias": len(noticias), "Pares": len(contagem_pares),
    "Grupos de sorteio": noticias["grupo"].nunique(),
    "Cópias além da primeira no texto original": noticias["texto"].duplicated().sum(),
    "Cópias além da primeira após limpeza": noticias["texto_limpo"].duplicated().sum(),
    "Textos vazios após limpeza": noticias["texto_limpo"].eq("").sum(),
}, name="Quantidade").to_frame()
display(auditoria)

pares_por_grupo = noticias.groupby("grupo")["id_par"].nunique().sort_index()
grupos_embaralhados = np.random.default_rng(semente).permutation(pares_por_grupo.index.to_numpy())
pares_restantes = 720
grupos_teste = set()
for grupo in grupos_embaralhados:
    tamanho_grupo = pares_por_grupo.loc[grupo]
    if tamanho_grupo <= pares_restantes:
        grupos_teste.add(grupo)
        pares_restantes -= tamanho_grupo
    if pares_restantes == 0:
        break
if pares_restantes:
    raise ValueError("A divisão não atingiu 720 pares de teste.")

noticias["particao"] = np.where(noticias["grupo"].isin(grupos_teste), "Teste", "Treino")
treino = noticias.loc[noticias["particao"].eq("Treino")].sort_values(["id_par", "classe"])
teste = noticias.loc[noticias["particao"].eq("Teste")].sort_values(["id_par", "classe"])
assert len(treino) == 5760 and len(teste) == 1440
assert set(treino["id_par"]).isdisjoint(teste["id_par"])
assert set(treino["texto_limpo"]).isdisjoint(teste["texto_limpo"])

divisao = noticias.groupby("particao").agg(
    Notícias=("id_par", "size"), Pares=("id_par", "nunique"), Grupos=("grupo", "nunique"),
).join(pd.crosstab(noticias["particao"], noticias["rotulo"]).rename(
    columns={"fake": "Falsas", "true": "Verdadeiras"}
)).reindex(["Treino", "Teste"])
display(divisao)

## 3. TF-IDF e os três classificadores

O TF-IDF dá mais destaque a termos frequentes naquela notícia e relativamente raros no conjunto de textos. Usamos palavras individuais, até 5.000 termos e normalização L2. **O vocabulário e o IDF são ajustados somente no treino.** Depois aplicamos a mesma transformação ao teste.

| Classificador | Como decide | Configuração |
|---|---|---|
| SVM linear | Aprende uma fronteira linear com margem entre as classes | C = 1 |
| Naive Bayes multinomial | Combina evidências dos termos estimadas por classe | Suavização α = 1 |
| Regressão logística | Estima a probabilidade de a notícia pertencer à classe falsa | C = 1, regularização L2 padrão |

A avaliação usa as predições padrão dos modelos. Para as curvas que variam o limiar, usamos o escore de decisão do SVM e a probabilidade da classe falsa do Naive Bayes e da regressão logística. Não há busca de hiperparâmetros neste experimento.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), norm="l2")
X_treino = tfidf.fit_transform(treino["texto_limpo"])
X_teste = tfidf.transform(teste["texto_limpo"])
y_treino = treino["classe"].to_numpy()
y_teste = teste["classe"].to_numpy()

modelos = {
    "SVM linear": LinearSVC(C=1, dual=False, max_iter=10000, random_state=semente),
    "Naive Bayes": MultinomialNB(alpha=1),
    "Regressão logística": LogisticRegression(C=1, solver="lbfgs", max_iter=2000),
}
predicoes = {}
escores = {}
for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    predicoes[nome] = modelo.predict(X_teste)
    escores[nome] = (
        modelo.decision_function(X_teste) if nome == "SVM linear"
        else modelo.predict_proba(X_teste)[:, 1]
    )
print(f"TF-IDF – treino: {X_treino.shape} | teste: {X_teste.shape}")
print("Modelos treinados:", ", ".join(modelos))

## 4. Métricas no conjunto de teste

- **Acurácia** é a proporção de acertos. **Acurácia balanceada** é a média dos recalls das duas classes e coincide com a acurácia neste teste balanceado.
- **Precisão das falsas** é a proporção de notícias realmente falsas entre as sinalizadas. **Recall das falsas** é a proporção de falsas encontradas. **Especificidade** é o recall das verdadeiras.
- **F1 macro** é a média simples do F1 das duas classes. Ele é a métrica principal. A tabela por classe mostra precisão, recall e F1 separadamente.
- **AUC-ROC** e **AP** (precisão média) resumem a ordenação dos escores da classe falsa ao variar o limiar. **MCC** mede a associação entre previsto e real, de −1 a 1.

In [ ]:
resultados = []
por_classe_linhas = []
matrizes = {}
for nome, previsto in predicoes.items():
    matriz = confusion_matrix(y_teste, previsto, labels=[0, 1])
    matrizes[nome] = matriz
    resultados.append({
        "Modelo": nome,
        "Acurácia": accuracy_score(y_teste, previsto),
        "Acurácia balanceada": balanced_accuracy_score(y_teste, previsto),
        "Precisão falsas": precision_score(y_teste, previsto, zero_division=0),
        "Recall falsas": recall_score(y_teste, previsto, zero_division=0),
        "Especificidade": recall_score(y_teste, previsto, pos_label=0, zero_division=0),
        "F1 macro": f1_score(y_teste, previsto, average="macro", zero_division=0),
        "AUC-ROC": roc_auc_score(y_teste, escores[nome]),
        "AP": average_precision_score(y_teste, escores[nome]),
        "MCC": matthews_corrcoef(y_teste, previsto),
        "Falsos positivos": matriz[0, 1],
        "Falsos negativos": matriz[1, 0],
    })
    relatorio = classification_report(
        y_teste, previsto, labels=[0, 1], target_names=["Verdadeira", "Falsa"],
        output_dict=True, zero_division=0,
    )
    for classe in ("Verdadeira", "Falsa"):
        r = relatorio[classe]
        por_classe_linhas.append({
            "Modelo": nome, "Classe": classe, "Precisão": r["precision"],
            "Recall": r["recall"], "F1": r["f1-score"], "Amostras": int(r["support"]),
        })

metricas = pd.DataFrame(resultados).set_index("Modelo").sort_values("F1 macro", ascending=False)
por_classe = pd.DataFrame(por_classe_linhas)
principais = ["F1 macro", "Acurácia", "Precisão falsas", "Recall falsas", "Especificidade"]
complementares = ["Acurácia balanceada", "AUC-ROC", "AP",
                 "MCC", "Falsos positivos", "Falsos negativos"]
def mostrar_tabela(tabela, formatos):
    exibicao = tabela.copy()
    for coluna, formato in formatos.items():
        exibicao[coluna] = exibicao[coluna].map(
            lambda valor: formato.format(valor) if pd.notna(valor) else "—"
        )
    display(exibicao)

mostrar_tabela(metricas[principais], dict.fromkeys(principais, "{:.2%}"))
mostrar_tabela(metricas[complementares], {
    "Acurácia balanceada": "{:.2%}", "AUC-ROC": "{:.2%}", "AP": "{:.2%}", "MCC": "{:.3f}",
})
mostrar_tabela(por_classe, dict.fromkeys(["Precisão", "Recall", "F1"], "{:.2%}"))

## 5. Gráficos e matrizes de confusão

As matrizes mostram o rótulo real nas linhas e a previsão nas colunas. **Falso positivo** é uma notícia verdadeira que foi sinalizada. **Falso negativo** é uma notícia falsa que passou. As três matrizes usam a mesma escala de cores.

O gráfico de pontos amplia a faixa onde estão as métricas, mantendo o eixo em porcentagem. Nas curvas, a linha tracejada indica o desempenho de referência aleatória.

In [ ]:
saida = Path("resultados_fakebr")
saida.mkdir(exist_ok=True)
cores = dict(zip(modelos, ("#185D75", "#AB4B23", "#4C7743")))

def mostrar_figura(fig, nome):
    fig.savefig(saida / nome, dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

fig, ax = plt.subplots(figsize=(9.2, 4.4), layout="constrained")
indicadores = ["F1 macro", "Acurácia", "Precisão falsas", "Recall falsas", "AUC-ROC"]
posicoes = np.arange(len(indicadores))
for i, nome in enumerate(modelos):
    ax.plot(metricas.loc[nome, indicadores], posicoes + (i - 1) * .20,
            "o", color=cores[nome], label=nome, ms=7)
ax.set_yticks(posicoes, indicadores)
ax.invert_yaxis()
ax.set_xlim(max(0, metricas[indicadores].min().min() - .025), 1.003)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set(title="Desempenho nas mesmas 1.440 notícias", xlabel="Desempenho (%)")
ax.grid(axis="x", alpha=.25)
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.17), ncol=3, frameon=False)
mostrar_figura(fig, "comparacao_metricas.png")

fig, eixos = plt.subplots(1, 3, figsize=(11.2, 4), layout="constrained")
for ax, (nome, matriz) in zip(eixos, matrizes.items()):
    ConfusionMatrixDisplay(matriz, display_labels=["Verdadeira", "Falsa"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d",
        im_kw={"vmin": 0, "vmax": 720},
    )
    ax.set(title=nome, xlabel="Classe prevista", ylabel="Classe real")
fig.suptitle("Matrizes de confusão no teste")
mostrar_figura(fig, "matrizes_confusao.png")

fig, (roc, pr) = plt.subplots(1, 2, figsize=(11.2, 4.5), layout="constrained")
for nome, escore in escores.items():
    fpr, tpr, _ = roc_curve(y_teste, escore)
    precisao, recall, _ = precision_recall_curve(y_teste, escore)
    roc.plot(fpr, tpr, color=cores[nome], label=f"{nome} ({metricas.loc[nome, 'AUC-ROC']:.3f})")
    pr.step(recall, precisao, where="post", color=cores[nome],
            label=f"{nome} ({metricas.loc[nome, 'AP']:.3f})")
roc.legend(loc="lower right", frameon=False)
pr.legend(loc="lower left", frameon=False)
roc.plot([0, 1], [0, 1], "--", color="#666666", linewidth=.8)
pr.axhline(y_teste.mean(), linestyle="--", color="#666666", linewidth=.8)
roc.set(title="Curva ROC", xlabel="Taxa de falsos positivos", ylabel="Taxa de verdadeiros positivos")
pr.set(title="Curva de precisão e recall", xlabel="Recall das falsas", ylabel="Precisão das falsas")
mostrar_figura(fig, "curvas_roc_pr.png")

## 6. Hipótese, tamanho de efeito e incerteza

O efeito é a diferença de **F1 macro em pontos percentuais**. Calculamos um intervalo individual de 95% para cada diferença com 10.000 reamostragens bootstrap dos grupos de teste. Cada reamostragem preserva os pares e usa os mesmos grupos para os três modelos.

Para a hipótese principal, testamos se o escore observado favorece o SVM frente ao Naive Bayes. A hipótese nula operacional é a intercambiabilidade das previsões dos dois modelos dentro de cada grupo. Em cada uma das 10.000 permutações, sorteamos em quais grupos trocaremos todas as previsões de um modelo com as do outro. O p-valor unilateral conta diferenças pelo menos tão grandes quanto a observada, com correção de +1. Usamos α = 0,05. As duas comparações com regressão logística são descritivas, com intervalos individuais.

A inferência supõe independência entre os grupos e, no teste, intercambiabilidade sob a hipótese nula. Ela descreve a incerteza para os modelos já treinados, na amostra de teste. Notícias de um mesmo veículo ou sobre o mesmo fato podem ter dependência residual.

In [ ]:
def f1_de_contagens(contagens):
    vn, fp, fn, vp = np.moveaxis(contagens, -1, 0)
    return .5 * (2 * vn / (2 * vn + fp + fn) + 2 * vp / (2 * vp + fp + fn))

indice_grupo, grupos = pd.factorize(teste["grupo"], sort=True)
por_grupo = {}
for nome, previsto in predicoes.items():
    contagens = np.zeros((len(grupos), 4), dtype=np.int32)
    np.add.at(contagens, (indice_grupo, 2 * y_teste + previsto), 1)
    por_grupo[nome] = contagens
    assert np.array_equal(contagens.sum(axis=0), matrizes[nome].ravel())
    assert np.isclose(f1_de_contagens(contagens.sum(axis=0)), metricas.loc[nome, "F1 macro"])

comparacoes = [
    ("SVM linear", "Naive Bayes"), ("SVM linear", "Regressão logística"),
    ("Regressão logística", "Naive Bayes"),
]
repeticoes = 10_000
alpha = 0.05
rng_boot = np.random.default_rng(semente + 1)
rng_perm = np.random.default_rng(semente + 2)
boot = {par: [] for par in comparacoes}

svm = por_grupo["SVM linear"]
bayes = por_grupo["Naive Bayes"]
total_svm, total_bayes = svm.sum(axis=0), bayes.sum(axis=0)
efeito = f1_de_contagens(total_svm) - f1_de_contagens(total_bayes)
extremos = 0
for inicio in range(0, repeticoes, 250):
    lote = min(250, repeticoes - inicio)
    ids = rng_boot.integers(len(grupos), size=(lote, len(grupos)))
    f1_amostras = {nome: f1_de_contagens(contagens[ids].sum(axis=1))
                  for nome, contagens in por_grupo.items()}
    for a, b in comparacoes:
        boot[a, b].extend(f1_amostras[a] - f1_amostras[b])

    trocar = rng_perm.integers(0, 2, size=(lote, len(grupos)), dtype=np.int32)
    ajuste = trocar @ (svm - bayes)
    permutado = f1_de_contagens(total_svm - ajuste) - f1_de_contagens(total_bayes + ajuste)
    extremos += np.count_nonzero(permutado >= efeito - 1e-12)

p_valor = (extremos + 1) / (repeticoes + 1)
linhas = []
for i, (a, b) in enumerate(comparacoes):
    baixo, alto = 100 * np.quantile(boot[a, b], [.025, .975])
    linhas.append({
        "Comparação": f"{a} − {b}",
        "Diferença (p.p.)": 100 * (metricas.loc[a, "F1 macro"] - metricas.loc[b, "F1 macro"]),
        "IC 95% inferior": baixo,
        "IC 95% superior": alto,
        "p unilateral": p_valor if i == 0 else np.nan,
    })

inferencia = pd.DataFrame(linhas)
mostrar_tabela(inferencia, {
    "Diferença (p.p.)": "{:+.2f}", "IC 95% inferior": "{:+.2f}",
    "IC 95% superior": "{:+.2f}", "p unilateral": "{:.4f}",
})

fig, ax = plt.subplots(figsize=(9, 3.4), layout="constrained")
for i, linha in inferencia.iterrows():
    ax.hlines(i, linha["IC 95% inferior"], linha["IC 95% superior"], color="#185D75", lw=2)
    ax.plot(linha["Diferença (p.p.)"], i, "o", color="#185D75")
ax.axvline(0, color="#666666", linestyle="--", linewidth=1)
ax.set_yticks(range(len(inferencia)), inferencia["Comparação"])
ax.invert_yaxis()
ax.set(title="Diferenças no F1 macro", xlabel="Diferença em pontos percentuais · IC de 95%")
ax.grid(axis="x", alpha=.25)
mostrar_figura(fig, "diferencas_f1.png")

## 7. O que apresentar

1. **Pergunta e hipótese.** O desenho compara três classificadores mantendo o mesmo TF-IDF.
2. **Dados.** Mostre a divisão 80/20 e explique por que cada par ficou junto e como o TF-IDF foi ajustado só no treino.
3. **Resultado.** Mostre a tabela de F1 macro, as matrizes de confusão e o intervalo da diferença principal. Use os falsos negativos e falsos positivos para explicar a consequência prática da escolha de um modelo.
4. **Limitações.** A avaliação usou textos completos de um único corpus. Comprimento, época e estilo dos veículos podem influenciar os escores. A base também disponibiliza textos de comprimento igualado por par para estudos futuros. Modelos de texto ajudam na triagem e sinalizam exemplos para revisão humana.

Almeida et al. (2021) incluíram o SVM linear e o Naive Bayes multinomial com TF-IDF, mas usaram um recorte político no treino e um teste externo próprio. Aqui, a regressão logística é o terceiro modelo e o teste é feito dentro do próprio Fake.Br. Os números dos dois experimentos correspondem a amostras diferentes. [Artigo de Almeida et al.](https://doi.org/10.5753/wics.2021.15966)

In [ ]:
def pt(valor, casas=2):
    return f"{valor:.{casas}f}".replace(".", ",")

vencedor = metricas.index[0]
principal = inferencia.iloc[0]
print(f"Maior F1 macro no teste: {vencedor}, {pt(100 * metricas.loc[vencedor, 'F1 macro'])}%.")
for nome, linha in metricas.iterrows():
    print(f"{nome}: {int(linha['Falsos negativos'])} falsas passaram e "
          f"{int(linha['Falsos positivos'])} verdadeiras foram sinalizadas.")
print(f"SVM − Naive Bayes: {pt(principal['Diferença (p.p.)'])} p.p., "
      f"IC 95% [{pt(principal['IC 95% inferior'])}; {pt(principal['IC 95% superior'])}] p.p.")
print(f"p unilateral = {pt(p_valor, 4)} (α = 0,05).")
if p_valor <= alpha:
    print("A hipótese nula operacional foi rejeitada. A diferença favoreceu o SVM.")
else:
    print("A hipótese nula operacional não foi rejeitada no teste de superioridade do SVM.")

## 8. Exportar resultados para a apresentação

Na aba **Arquivos** do Colab, baixe **`MCEQ_Resultados_FakeBr.zip`** após executar a célula. O ZIP inclui tabelas em CSV, as quatro figuras em PNG, as três matrizes em CSV, previsões por notícia, as partições e dados de reprodução.

**Referências técnicas:** [TF-IDF e pré-processamento](https://scikit-learn.org/stable/common_pitfalls.html) · [Métricas de classificação](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics) · [Princípio do teste de permutação](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html)

In [ ]:
metricas.to_csv(saida / "metricas.csv")
por_classe.to_csv(saida / "metricas_por_classe.csv", index=False)
divisao.to_csv(saida / "divisao.csv")
auditoria.to_csv(saida / "auditoria.csv")
inferencia.to_csv(saida / "comparacoes_f1.csv", index=False)
noticias[["id_par", "grupo", "rotulo", "particao"]].to_csv(saida / "particoes.csv", index=False)

previsoes = teste[["id_par", "grupo", "rotulo", "classe"]].copy()
for nome, chave in zip(modelos, ("svm", "nb", "logistica")):
    previsoes[f"previsao_{chave}"] = predicoes[nome]
    previsoes[f"escore_falsa_{chave}"] = escores[nome]
    pd.DataFrame(
        matrizes[nome], index=["Real verdadeira", "Real falsa"],
        columns=["Prevista verdadeira", "Prevista falsa"],
    ).to_csv(saida / f"matriz_{chave}.csv")
previsoes.to_csv(saida / "previsoes_teste.csv", index=False)

config = {
    "revisao_fakebr": revisao, "versao_dos_textos": "full_texts", "classe_positiva": "fake",
    "semente": semente, "proporcao_teste": .20, "metrica_principal": "F1 macro",
    "repeticoes": repeticoes, "alpha": alpha, "python": platform.python_version(),
    "scikit_learn": sklearn.__version__, "pandas": pd.__version__,
    "numpy": np.__version__, "matplotlib": matplotlib.__version__,
}
(saida / "reprodutibilidade.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
arquivo_zip = shutil.make_archive("MCEQ_Resultados_FakeBr", "zip", saida)
print(f"Arquivo gerado: {Path(arquivo_zip).name}")